# Grid IoT RL — Multi-Device Simulator

Runs 3 edge devices simultaneously using Python threads.
Each device sends to AWS IoT Core using its own certificate.

**Before running:** upload this entire `colab/` folder to Google Drive with cert files inside `certs/` subfolders.

In [ ]:
!pip install awsiotsdk awscrt boto3 -q
print('Dependencies installed')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
# ── CONFIGURE HERE ────────────────────────────────────────────
DRIVE_FOLDER = '/content/drive/MyDrive/colab'  # path to colab/ folder in Drive
IOT_ENDPOINT = 'YOUR-ENDPOINT-ats.iot.us-east-1.amazonaws.com'

DEVICES = [
    {'id': 'edge-device-001', 'mode': 'happy',   'fault_rate': 0.1, 'count': 300},
    {'id': 'edge-device-002', 'mode': 'noisy',   'fault_rate': 0.2, 'count': 300},
    {'id': 'edge-device-003', 'mode': 'network', 'fault_rate': 0.3, 'count': 300},
]
print('Configuration set')

In [ ]:
import os, shutil, sys

# copy all files from Drive to Colab runtime
for item in ['data_generator.py','publish_to_iot.py','requirements.txt']:
    src = os.path.join(DRIVE_FOLDER, item)
    if os.path.exists(src):
        shutil.copy(src, f'/content/{item}')

os.makedirs('/content/modules', exist_ok=True)
for mod in ['__init__.py','noise_injector.py','network_impairments.py','attack_simulator.py']:
    src = os.path.join(DRIVE_FOLDER, 'modules', mod)
    if os.path.exists(src):
        shutil.copy(src, f'/content/modules/{mod}')

for d in ['edge-device-001','edge-device-002','edge-device-003']:
    os.makedirs(f'/content/certs/{d}', exist_ok=True)
    for f in ['cert.pem','private.key']:
        src = os.path.join(DRIVE_FOLDER, 'certs', d, f)
        if os.path.exists(src):
            shutil.copy(src, f'/content/certs/{d}/{f}')

root_ca_src = os.path.join(DRIVE_FOLDER, 'certs', 'root-CA.crt')
if os.path.exists(root_ca_src):
    shutil.copy(root_ca_src, '/content/certs/root-CA.crt')

os.chdir('/content')
print('Files ready')
print('Certs:', os.listdir('/content/certs'))

In [ ]:
import threading, subprocess, sys

def run_device(device):
    cmd = [
        sys.executable, 'publish_to_iot.py',
        '--endpoint',    IOT_ENDPOINT,
        '--cert',        f'/content/certs/{device["id"]}/cert.pem',
        '--key',         f'/content/certs/{device["id"]}/private.key',
        '--root-ca',     '/content/certs/root-CA.crt',
        '--client-id',   device['id'],
        '--count',       str(device['count']),
        '--fault-rate',  str(device['fault_rate']),
        '--mode',        device['mode'],
        '--interval',    '1.0',
    ]
    print(f"Starting {device['id']} ({device['mode']} mode)")
    subprocess.run(cmd)
    print(f"Done: {device['id']}")

threads = [threading.Thread(target=run_device, args=(d,)) for d in DEVICES]
for t in threads: t.start()
print(f'{len(threads)} devices running simultaneously...')
for t in threads: t.join()
print('All devices finished')

In [ ]:
# Verify data landed in S3
import boto3
s3 = boto3.client('s3')
buckets = s3.list_buckets()['Buckets']
print('S3 buckets:', [b['Name'] for b in buckets])

# check latest readings
for b in buckets:
    if 'grid' in b['Name'].lower():
        resp = s3.list_objects_v2(Bucket=b['Name'], Prefix='readings/', MaxKeys=5)
        print(f"\nBucket: {b['Name']}")
        for obj in resp.get('Contents', []):
            print(f"  {obj['Key']} ({obj['Size']} bytes)")

## Delete SageMaker endpoint when done

```python
import boto3
boto3.client('sagemaker').delete_endpoint(EndpointName='grid-voltage-rl-v1')
```